In [1]:
import os 
import sys 
import numpy as np 
import torch 
import matplotlib.pyplot as plt
from tqdm import tqdm 
import time 
import random 

parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)

from src.dataset.custom_dataset import GenDataset, DiscDataset
from src.model.graph_model import NEGATGenerator, DiffPoolDiscriminator
from src.training.trainer import train_GAN

from utils.gen_utils import load_config, get_device, dataset_splitter, generate_markdown_report_GAN_and_save_model
from utils.ppnet_utils import initialize_network
from utils.load_data_utils import load_sampled_input_data

yaml_config= load_config('config_gan.yaml')

device = get_device(yaml_config['device'])

net = initialize_network(net_name=yaml_config['data']['net_name'],
                        #  net_name=yaml_config['data']['net_name'], 
                         load_std=yaml_config['data']['load_std']) 

########### data for generator 
sampled_input_data_G = load_sampled_input_data(sc_type=yaml_config['data']['gen_scenario_type'], 
                                               net=net, 
                                               num_samples=yaml_config['data']['num_samples'], 
                                               noise=yaml_config['data']['noise'])

# sparsity of the node and edge features 
node_feat_sparsity = np.count_nonzero(sampled_input_data_G['node_mask']) / sampled_input_data_G['node_mask'].numpy().size
pflow_edge_sparsity = np.count_nonzero(sampled_input_data_G['edge_mask'][:,:,0]) / sampled_input_data_G['edge_mask'][:,:,0].numpy().size

print(f"Sparsity of PV measurements at buses = {node_feat_sparsity:.1f}%")
print(f"Sparsity of P+ measurements at branches = {pflow_edge_sparsity:.1f}")

########### data for discriminator 
sampled_input_data_D = load_sampled_input_data(sc_type=yaml_config['data']['dis_scenario_type'], 
                                               net=net, 
                                               num_samples=yaml_config['data']['num_samples'], 
                                               noise=yaml_config['data']['noise'])


dataset_G = GenDataset(model_name=yaml_config['model_G']['name'], 
                       sampled_input_data=sampled_input_data_G)

(train_loader_G, val_loader_G, test_loader_G), _ = dataset_splitter(dataset_G, 
                                                                    batch_size=yaml_config['loader']['batch_size'])

dataset_D = DiscDataset(sampled_input_data=sampled_input_data_D)

(train_loader_D, val_loader_D, test_loader_D), _ = dataset_splitter(dataset_D,
                                                                    batch_size=yaml_config['loader']['batch_size'])

###########################################################
seeds = np.arange(100)


all_losses_seeds = {}
generated_data = {}
simulated_v_pf_data = {}
all_time_counters = {}


for seed in tqdm(seeds):
    
    random.seed(int(seed))
    np.random.seed(seed)
    torch.manual_seed(seed)

    start_time_training = time.perf_counter()
    # instantiate model, optimizer and schedular for Generator 
    model_G = NEGATGenerator(node_input_features=dataset_G[0][0].x.shape[-1], 
                        list_node_hidden_features=yaml_config['model_G']['list_node_hidden_features'], # [128,64], 
                        node_out_features=yaml_config['model_G']['node_out_features'], # 64, 
                        k_hop_node=yaml_config['model_G']['k_hop_node'], #1, 
                        edge_input_features=dataset_G[0][1].x.shape[-1], 
                        list_edge_hidden_features=yaml_config['model_G']['list_edge_hidden_features'], #[128,64], 
                        edge_output_features=yaml_config['model_G']['edge_out_features'], #64, 
                        k_hop_edge=yaml_config['model_G']['k_hop_edge'], #1, 
                        gat_out_features=yaml_config['model_G']['gat_out_features'], #32, 
                        gat_head=yaml_config['model_G']['gat_head'], #2, 
                        device=device)

    optimizer_G = torch.optim.Adam(model_G.parameters(), 
                                lr=yaml_config['training_G']['lr'], 
                                weight_decay=yaml_config['training_G']['weight_decay'])

    schedular_G = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer_G, 
                                                        mode='min', 
                                                        factor=0.1, 
                                                        min_lr=yaml_config['training_G']['schedular_min_lr'])

    total_params_G = sum(p.numel() for p in model_G.parameters() if p.requires_grad)
    print(f'Total number of parameters of model {model_G}: {total_params_G}')

    # instantiate model, optimizer and schedular for Discriminator 
    model_D = DiffPoolDiscriminator(in_channel=dataset_D[0].x.shape[-1], 
                                hidden_channel=yaml_config['model_D']['hidden_channel'], 
                                out_channel=yaml_config['model_D']['out_channel'], 
                                num_nodes=len(net.bus.index))

    total_params_D = sum(p.numel() for p in model_D.parameters() if p.requires_grad)
    print(f'Total number of parameters of model {model_D}: {total_params_D}')

    optimizer_D = torch.optim.Adam(model_D.parameters(), 
                                lr=yaml_config['training_D']['lr'], 
                                weight_decay=yaml_config['training_D']['weight_decay'])

    schedular_D = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer_D, 
                                                        mode='min', 
                                                        factor=0.1, 
                                                        min_lr=yaml_config['training_D']['schedular_min_lr'])
    
    all_losses_seeds[seed] = train_GAN(model_G=model_G, 
                                        model_D=model_D, 
                                        all_loader_G= [train_loader_G, val_loader_G, test_loader_G], 
                                        all_loader_D= [train_loader_D, val_loader_D, test_loader_D],  
                                        optimizer_G=optimizer_G, 
                                        optimizer_D=optimizer_D, 
                                        schedular_G=schedular_G, 
                                        schedular_D=schedular_D, 
                                        num_epoch=yaml_config['training_GAN']['num_epoch'], 
                                        disc_iter=yaml_config['training_GAN']['disc_iter'], 
                                        gen_iter=yaml_config['training_GAN']['gen_iter'],
                                        feature_matching=yaml_config['training_GAN']['feature_matching'],   
                                        device=device)
    try: 
        generated_data[seed], simulated_v_pf_data[seed] = generate_markdown_report_GAN_and_save_model(yaml_config=yaml_config, 
                                                                        train_g_losses=all_losses_seeds[seed]['train_g_losses'], 
                                                                        train_d_losses=all_losses_seeds[seed]['train_d_losses'], 
                                                                        train_d_accuracies=all_losses_seeds[seed]['train_d_accuracies'], 
                                                                        parent_dir=parent_dir, 
                                                                        test_loader_G=test_loader_G, 
                                                                        model_G=model_G, 
                                                                        sampled_input_data_G=sampled_input_data_G, 
                                                                        return_data=True,
                                                                        seed=seed)

    except Exception as e: 
        print(f"At seed {seed}: ")
        generated_data[seed] = 0
        simulated_v_pf_data[seed] = 0
    
    end_time_training = time.perf_counter() 

    all_time_counters[seed] = end_time_training - start_time_training


Network: net_A is selected 

Net net_A has 42 nodes and 42 edges. 

Selecting all the MV/LV transformers in the network 

Scaling inputs...
Number of V, P measurements 40 out of 84

Number of P_to, Q_to, P_from, Q_from measurements 23 out of 42

Sparsity of PV measurements at buses = 0.5%
Sparsity of P+ measurements at branches = 0.5
Scaling inputs...
Dataset for NEGATGenerator selected!


 Directed power flows accounted in dataset...


 get_edge_index_lu handling dictionary of tensors...



  0%|          | 0/100 [00:00<?, ?it/s]

Total number of parameters of model NEGATGenerator(
  (node_layers): ModuleList(
    (0): TAGConv(2, 64, K=3)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (fc_node): Linear(in_features=64, out_features=32, bias=True)
  (edge_layers): ModuleList(
    (0): ModuleList(
      (0-1): 2 x TAGConv(6, 128, K=1)
    )
  )
  (edge_biases): ParameterList(  (0): Parameter containing: [torch.float32 of size 128])
  (fc_edge): Linear(in_features=128, out_features=1, bias=True)
  (gatconv): GATConv(32, 32, heads=1)
  (mlp_gat): Sequential(
    (0): Linear(in_features=32, out_features=2, bias=True)
  )
): 7363
Total number of parameters of model DiffPoolDiscriminator(): 2667
At epoch: 0
training: D = 1.564e+00, Acc = 0.70, G = 2.736e+01
validation: D = 9.812e-01, Acc = 0.70, G = 2.218e+01, lr_D = 1.0e-05, lr_G 1.0e-04
------------------------------------------------------------------------------
At epoch: 2
training: D = 1.093e+00, Acc = 0.65, G = 8.922e+00
validation: D = 7.748

  1%|          | 1/100 [01:17<2:08:11, 77.69s/it]

📄 Training report saved to: /Users/sohamprajapati/Documents/Thesis/Project Codes/TapSEGNN/results/GAN_only/20251002_211842_0/training_report.md
Total number of parameters of model NEGATGenerator(
  (node_layers): ModuleList(
    (0): TAGConv(2, 64, K=3)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (fc_node): Linear(in_features=64, out_features=32, bias=True)
  (edge_layers): ModuleList(
    (0): ModuleList(
      (0-1): 2 x TAGConv(6, 128, K=1)
    )
  )
  (edge_biases): ParameterList(  (0): Parameter containing: [torch.float32 of size 128])
  (fc_edge): Linear(in_features=128, out_features=1, bias=True)
  (gatconv): GATConv(32, 32, heads=1)
  (mlp_gat): Sequential(
    (0): Linear(in_features=32, out_features=2, bias=True)
  )
): 7363
Total number of parameters of model DiffPoolDiscriminator(): 2667
At epoch: 0
training: D = 1.816e+00, Acc = 0.73, G = 2.961e+01
validation: D = 8.904e-01, Acc = 0.70, G = 2.152e+01, lr_D = 1.0e-05, lr_G 1.0e-04
-------------------

...

📄 Training report saved to: /Users/sohamprajapati/Documents/Thesis/Project Codes/TapSEGNN/results/GAN_only/20251002_232029_97/training_report.md
Total number of parameters of model NEGATGenerator(
  (node_layers): ModuleList(
    (0): TAGConv(2, 64, K=3)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (fc_node): Linear(in_features=64, out_features=32, bias=True)
  (edge_layers): ModuleList(
    (0): ModuleList(
      (0-1): 2 x TAGConv(6, 128, K=1)
    )
  )
  (edge_biases): ParameterList(  (0): Parameter containing: [torch.float32 of size 128])
  (fc_edge): Linear(in_features=128, out_features=1, bias=True)
  (gatconv): GATConv(32, 32, heads=1)
  (mlp_gat): Sequential(
    (0): Linear(in_features=32, out_features=2, bias=True)
  )
): 7363
Total number of parameters of model DiffPoolDiscriminator(): 2667
At epoch: 0
training: D = 5.106e-01, Acc = 0.75, G = 3.339e+01
validation: D = 3.858e-01, Acc = 0.75, G = 1.974e+01, lr_D = 1.0e-05, lr_G 1.0e-04
------------------

 99%|█████████▉| 99/100 [2:04:19<01:14, 74.43s/it]

📄 Training report saved to: /Users/sohamprajapati/Documents/Thesis/Project Codes/TapSEGNN/results/GAN_only/20251002_232143_98/training_report.md
Total number of parameters of model NEGATGenerator(
  (node_layers): ModuleList(
    (0): TAGConv(2, 64, K=3)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (fc_node): Linear(in_features=64, out_features=32, bias=True)
  (edge_layers): ModuleList(
    (0): ModuleList(
      (0-1): 2 x TAGConv(6, 128, K=1)
    )
  )
  (edge_biases): ParameterList(  (0): Parameter containing: [torch.float32 of size 128])
  (fc_edge): Linear(in_features=128, out_features=1, bias=True)
  (gatconv): GATConv(32, 32, heads=1)
  (mlp_gat): Sequential(
    (0): Linear(in_features=32, out_features=2, bias=True)
  )
): 7363
Total number of parameters of model DiffPoolDiscriminator(): 2667
At epoch: 0
training: D = 2.414e+00, Acc = 0.41, G = 9.054e+01
validation: D = 2.103e+00, Acc = 0.39, G = 4.441e+01, lr_D = 1.0e-05, lr_G 1.0e-04
------------------

100%|██████████| 100/100 [2:05:34<00:00, 75.34s/it]

📄 Training report saved to: /Users/sohamprajapati/Documents/Thesis/Project Codes/TapSEGNN/results/GAN_only/20251002_232258_99/training_report.md


In [2]:
# import joblib 
# joblib.dump(generated_data, parent_dir + f'/results/GAN_only/Oct2_all_seed_generated_data.pkl')
# joblib.dump(all_losses_seeds, parent_dir + f'/results/GAN_only/Oct2_all_seed_losses_data.pkl')
# joblib.dump(simulated_v_pf_data, parent_dir + f'/results/GAN_only/Oct2__all_seed_simulated_v_pf_data.pkl')
# joblib.dump(all_time_counters, parent_dir + f'/results/GAN_only/Oct2_all_time_counters.pkl')

['/Users/sohamprajapati/Documents/Thesis/Project Codes/TapSEGNN/results/GAN_only/Oct2_all_time_counters.pkl']

In [4]:
import pandas as pd 
# plot the variance of simulated vs. generated voltages 
generated_v_df = pd.DataFrame(columns=seeds)
simulated_v_pf_data_df = pd.DataFrame(columns=seeds)
all_val_g_losses_df = pd.DataFrame(columns=seeds)
all_val_d_losses_df = pd.DataFrame(columns=seeds)
all_val_d_accuracies_df = pd.DataFrame(columns=seeds)

num_buses = len(net.bus)

for seed in seeds: 
    all_val_g_losses_df[seed] = all_losses_seeds[seed]['val_g_losses']
    all_val_d_losses_df[seed] = all_losses_seeds[seed]['val_d_losses']
    all_val_d_accuracies_df[seed] = all_losses_seeds[seed]['val_d_accuracies']
    try: 
        generated_v_df[seed] = generated_data[seed]['gen_v'][:num_buses]
        simulated_v_pf_data_df[seed] = simulated_v_pf_data[seed]
    except Exception as e: 
        print(e)
generated_v_df.dropna(axis='columns', inplace=True)
simulated_v_pf_data_df.dropna(axis='columns', inplace=True)
# simulated_v_pf_data_df, generated_v_df

'int' object is not subscriptable
'int' object is not subscriptable
'int' object is not subscriptable
np.int64(25)
np.int64(26)
np.int64(27)
np.int64(28)
np.int64(29)
np.int64(30)
np.int64(31)
np.int64(32)
np.int64(33)
np.int64(34)
np.int64(35)
np.int64(36)
np.int64(37)
np.int64(38)
np.int64(39)
np.int64(40)
np.int64(41)
np.int64(42)
np.int64(43)
np.int64(44)
np.int64(45)
np.int64(46)
np.int64(47)
np.int64(48)
np.int64(49)
np.int64(50)
np.int64(51)
np.int64(52)
np.int64(53)
np.int64(54)
np.int64(55)
np.int64(56)
np.int64(57)
np.int64(58)
np.int64(59)
np.int64(60)
np.int64(61)
np.int64(62)
np.int64(63)
np.int64(64)
np.int64(65)
np.int64(66)
np.int64(67)
np.int64(68)
np.int64(69)
np.int64(70)
np.int64(71)
np.int64(72)
np.int64(73)
np.int64(74)
np.int64(75)
np.int64(76)
np.int64(77)
np.int64(78)
np.int64(79)
np.int64(80)
np.int64(81)
np.int64(82)
np.int64(83)
np.int64(84)
np.int64(85)
np.int64(86)
np.int64(87)
np.int64(88)
np.int64(89)
np.int64(90)
np.int64(91)
np.int64(92)
np.int64(93)
n

In [5]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Calculate mean and standard deviation for both dataframes
simulated_mean = simulated_v_pf_data_df.mean(axis=1)
simulated_std = simulated_v_pf_data_df.std(axis=1)
generated_mean = generated_v_df.mean(axis=1)
generated_std = generated_v_df.std(axis=1)

# Create the plot
plt.figure(figsize=(10, 6))
x = range(len(simulated_mean))
sns.lineplot(x=x, y=simulated_mean, label='Simulated')
plt.fill_between(x, simulated_mean - simulated_std, simulated_mean + simulated_std, alpha=0.3)
sns.lineplot(x=x, y=generated_mean, label='Generated')
plt.fill_between(x, generated_mean - generated_std, generated_mean + generated_std, alpha=0.3)
plt.xlabel('Bus')
plt.ylabel('Voltage [p.u]')
plt.legend()
plt.show()

/var/folders/5k/pz7fmgl977s51qnm4k4yx4400000gn/T/ipykernel_78468/3432040400.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
val_d_losses_mean = all_val_d_losses_df.mean(axis=1)
val_d_losses_std = all_val_d_losses_df.std(axis=1)

val_g_losses_mean = all_val_g_losses_df.mean(axis=1)
val_g_losses_std = all_val_g_losses_df.std(axis=1)

val_d_accuracies_mean = all_val_d_accuracies_df.mean(axis=1)
val_d_accuracies_std = all_val_d_accuracies_df.std(axis=1)

fig, ax = plt.subplots(3,1, figsize=(8,10))
x = range(len(val_d_losses_mean))

sns.lineplot(x=x, y=val_d_losses_mean, ax=ax[0], label='D Loss')
ax[0].fill_between(x, val_d_losses_mean - val_d_losses_std, val_d_losses_mean + val_d_losses_std, alpha=0.3)

sns.lineplot(x=x, y=val_g_losses_mean, ax=ax[1], label='G Loss')
ax[1].fill_between(x, val_g_losses_mean - val_g_losses_std, val_g_losses_mean + val_g_losses_std, alpha=0.3)

sns.lineplot(x=x, y=val_d_accuracies_mean, ax=ax[2], label='D Accuracy')
ax[2].fill_between(x, val_d_accuracies_mean - val_d_accuracies_std, val_d_accuracies_mean + val_d_accuracies_std, alpha=0.3)

plt.tight_layout()
plt.show()

/var/folders/5k/pz7fmgl977s51qnm4k4yx4400000gn/T/ipykernel_78468/3153675948.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# check which run has the least difference between simulated and generated 
gen_minus_sim = generated_v_df - simulated_v_pf_data_df
np.argmin(gen_minus_sim.abs().sum(axis=0))

np.int64(20)

In [8]:
np.min(gen_minus_sim.abs().sum(axis=0))

np.float64(0.16634474876408145)

In [9]:
gen_minus_sim.abs().sum(axis=0)

0     0.554146
2     0.634280
3     0.367697
4     1.005520
5     0.507404
6     0.584277
7     1.375584
9     0.526711
10    1.164113
11    0.551268
12    0.885495
13    0.768947
14    0.937414
15    0.899221
16    0.705154
17    0.591370
18    0.514632
19    0.757072
20    0.574498
21    0.276221
22    0.166345
24    1.415284
dtype: float64